# Task 2.4
Before training, verify your analytical gradients from Task 2.3 against numerical finite differences. For a randomly initialised network, compute both the analytical gradient $\frac{\partial L}{\partial W_{ij}^{(1)}}$ and the finite difference approximation:

$$
\frac{\partial L}{\partial W_{ij}^{(1)}} \approx
\frac{
L\left(W_{ij}^{(1)}+\epsilon\right)
-L\left(W_{ij}^{(1)}-\epsilon\right)
}{
2\epsilon
}
\tag{8}
$$

using $\epsilon=10^{-5}$

In [1]:
# %%capture
# %run Task2.ipynb

In [2]:
 # imports
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import math

## Equation
The two equations I have derived and will compare are:

**Gradient with respect to $W_n^{(1)}$:**

$$
\begin{aligned}
\frac{\partial L}{\partial W_n^{(1)}}
={}&
\frac{2}{N_d}\sum_{i=1}^{N_d}
\left(NN(t_i)-x(t_i)\right)
W_n^{(2)}t_i
\left(1-\tanh^2(z_{n,i})\right)
\\[6pt]
&+\lambda_{phys}\frac{2}{N_c}
\sum_{i=1}^{N_c}r_i
\Bigg[
-4W_n^{(2)}W_n^{(1)}
\tanh(z_{n,i})
\left(1-\tanh^2(z_{n,i})\right)
\\
&\quad
-2W_n^{(2)}(W_n^{(1)})^2t_i
\left(1-\tanh^2(z_{n,i})\right)^2
\\
&\quad
+4W_n^{(2)}(W_n^{(1)})^2t_i
\tanh^2(z_{n,i})
\left(1-\tanh^2(z_{n,i})\right)
\\
&\quad
+\frac{\mu}{m}
\Bigg(
W_n^{(2)}
\left(1-\tanh^2(z_{n,i})\right)
\\
&\qquad
-2W_n^{(2)}W_n^{(1)}t_i
\tanh(z_{n,i})
\left(1-\tanh^2(z_{n,i})\right)
\Bigg)
\\
&\quad
+\frac{k}{m}W_n^{(2)}t_i
\left(1-\tanh^2(z_{n,i})\right)
\Bigg]
\\[6pt]
&+\lambda_{IC}
\Bigg[
2\left(
\sum_{j=1}^{H}W_j^{(2)}W_j^{(1)}
\left(1-\tanh^2(b_j^{(1)})\right)
\right)
W_n^{(2)}
\left(1-\tanh^2(b_n^{(1)})\right)
\Bigg]
\end{aligned}
$$

**Gradient with respect to $W_n^{(2)}$:**

$$
\begin{aligned}
\frac{\partial L}{\partial W_n^{(2)}}
={}&
\frac{2}{N_d}\sum_{i=1}^{N_d}
\left(NN(t_i)-x(t_i)\right)
\tanh(z_{n,i})
\\[6pt]
&+\lambda_{phys}\frac{2}{N_c}
\sum_{i=1}^{N_c}r_i
\Bigg[
-2(W_n^{(1)})^2
\tanh(z_{n,i})
\left(1-\tanh^2(z_{n,i})\right)
\\
&\quad
+\frac{\mu}{m}W_n^{(1)}
\left(1-\tanh^2(z_{n,i})\right)
\\
&\quad
+\frac{k}{m}\tanh(z_{n,i})
\Bigg]
\\[6pt]
&+\lambda_{IC}
\Bigg[
2\left(
\sum_{j=1}^{H}W_j^{(2)}
\tanh(b_j^{(1)})+b^{(2)}-1
\right)
\tanh(b_n^{(1)})
\\
&\quad
+2\left(
\sum_{j=1}^{H}W_j^{(2)}W_j^{(1)}
\left(1-\tanh^2(b_j^{(1)})\right)
\right)
W_n^{(1)}
\left(1-\tanh^2(b_n^{(1)})\right)
\Bigg]
\end{aligned}
$$

**Loss function:**
$$
L = L_{data} + \lambda_{phys}\cdot L_{phys} + \lambda_{IC}\cdot L_{IC}
$$
**Loss function expanded:**

$$
\begin{aligned}
L ={}&
\frac{1}{N_d}\sum_{i=1}^{N_d}
\left(
NN(t_i)-e^{-2.5t_i}
\left(2\cdot0.5\cdot\cos(-0.11+22.22t_i)\right)
\right)^2
\\[6pt]
&+\lambda_{phys}\frac{1}{N_c}\sum_{i=1}^{N_c}
\left(
\frac{d^2NN}{dt^2}(t_i)
+\frac{\mu}{m}\frac{dNN}{dt}(t_i)
+\frac{k}{m}NN(t_i)
\right)^2
\\[6pt]
&+\lambda_{IC}
\left[
\left(NN(0)-1\right)^2
+\left(\frac{dNN}{dt}(0)\right)^2
\right]
\end{aligned}
$$

**Neural Network**
$$
NN(t_i) = \sum_{n=1}^{H} W_n^{(2)} \tanh\left(W_n^{(1)}t_i + b_n^{(1)}\right) + b^{(2)}
$$

**First derivative of the neural network:**

$$
\frac{dNN}{dt}(t_i) =
\sum_{n=1}^{H}
W_n^{(2)}W_n^{(1)}
\left(
1-\tanh^2\left(W_n^{(1)}t_i+b_n^{(1)}\right)
\right)
$$

**Second derivative of the neural network:**

$$
\frac{d^2NN}{dt^2}(t_i) =
-2\sum_{n=1}^{H}
W_n^{(2)}(W_n^{(1)})^2
\tanh\left(W_n^{(1)}t_i+b_n^{(1)}\right)
\left(
1-\tanh^2\left(W_n^{(1)}t_i+b_n^{(1)}\right)
\right)
$$

**Loss function with the neural network and its derivatives fully substituted:**

$$
\begin{aligned}
L ={}&
\frac{1}{N_d}\sum_{i=1}^{N_d}
\Bigg(
\sum_{n=1}^{H}
W_n^{(2)}
\tanh\left(W_n^{(1)}t_i+b_n^{(1)}\right)
+b^{(2)}
-e^{-2.5t_i}
\left(2\cdot0.5\cdot\cos(-0.11+22.22t_i)\right)
\Bigg)^2
\\[10pt]
&+\lambda_{phys}\frac{1}{N_c}
\sum_{i=1}^{N_c}
\Bigg[
-2\sum_{n=1}^{H}
W_n^{(2)}(W_n^{(1)})^2
\tanh\left(W_n^{(1)}t_i+b_n^{(1)}\right)
\left(
1-\tanh^2\left(W_n^{(1)}t_i+b_n^{(1)}\right)
\right)
\\[6pt]
&\qquad
+\frac{\mu}{m}
\sum_{n=1}^{H}
W_n^{(2)}W_n^{(1)}
\left(
1-\tanh^2\left(W_n^{(1)}t_i+b_n^{(1)}\right)
\right)
\\[6pt]
&\qquad
+\frac{k}{m}
\left(
\sum_{n=1}^{H}
W_n^{(2)}
\tanh\left(W_n^{(1)}t_i+b_n^{(1)}\right)
+b^{(2)}
\right)
\Bigg]^2
\\[10pt]
&+\lambda_{IC}
\Bigg[
\left(
\sum_{n=1}^{H}
W_n^{(2)}\tanh\left(b_n^{(1)}\right)
+b^{(2)}-1
\right)^2
\\[6pt]
&\qquad
+\left(
\sum_{n=1}^{H}
W_n^{(2)}W_n^{(1)}
\left(
1-\tanh^2\left(b_n^{(1)}\right)
\right)
\right)^2
\Bigg]
\end{aligned}
$$

Firstly, creating the neural network

In [10]:
#constants
m = 1.0 # mass
mu = 5 # friction constant
k = 500 # spring constant

omega_intial = np.sqrt(k)
delta = mu/(2*m)
omega = np.sqrt(omega_intial**2 - delta**2)

def A(phi):
    return 1 / (2*math.cos(phi))

phi = np.arctan(-1*delta/omega)

t_train = torch.linspace(start = 0, end = 0.5, steps = 10)
t_train = t_train.view(-1,1) # We transform it into shape (samples, features), instead of (samples). In our case that is (10,1)

def x(t):
    return  np.exp(-1*delta*t) *(2*A(phi)* np.cos(omega*t + phi))
y_train = x(t_train)

C:\Users\Saads\AppData\Local\Temp\ipykernel_13336\4019892306.py:19: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  return  np.exp(-1*delta*t) *(2*A(phi)* np.cos(omega*t + phi))


In [11]:
t_train = t_train.reshape(-1, 1)
t_train = t_train.detach().clone().requires_grad_(True).double()
t0 = torch.tensor([0.0], requires_grad=True)
# Creating the model
model = nn.Sequential(nn.Linear(1, 10), nn.Tanh(), nn.Linear(10, 1))
model = model.double()

# Making NN a function of t and then deriving it
# def NN(t):
#     return model(t)
# NN = model(t_train)
# dNN_dt = torch.autograd.grad(NN, t_train, grad_outputs=torch.ones_like(NN), create_graph=True)[0]
# dNN2_dt2 = torch.autograd.grad(dNN_dt, t_train, grad_outputs=torch.ones_like(dNN_dt), create_graph=True)[0]

# # Physics loss. The residuals must equal 0
# rt = dNN2_dt2 + (mu/m)*dNN_dt + (k/m)*NN
# loss_phys = torch.mean(rt**2)

# # Initial conditions loss. x(0) = 1 and x'(0) = 0
# NN_0 = model(t0)
# dNN_0 = torch.autograd.grad(NN_0, t0, create_graph=True)[0]
# loss_IC1 = torch.mean((NN_0 - 1)**2)
# loss_IC2 = torch.mean(dNN_0**2)

# # Data loss. The model must provide the same output as the closed form solution
# # Version 1
# # loss_data = []
# # for i in range(len(t_train)):
# #     data_loss_increment = torch.mean((model(t_train[i]) - y_train[i])**2)
# #     loss_data.append(data_loss_increment)
# # loss_data = torch.stack(loss_data).mean()
# # Version 2
# predictions = model(t_train)
# loss_fn = nn.MSELoss()
# assert predictions.shape == y_train.shape
# loss_data = loss_fn(predictions, y_train)


In [12]:
def calc_loss():
    t_col = torch.linspace(start = 0, end = 1, steps = 50).reshape(-1, 1).detach().clone().requires_grad_(True).double()
    NN = model(t_col)
    dNN_dt = torch.autograd.grad(NN, t_col, grad_outputs=torch.ones_like(NN), create_graph=True)[0]
    dNN2_dt2 = torch.autograd.grad(dNN_dt, t_col, grad_outputs=torch.ones_like(dNN_dt), create_graph=True)[0]

    # Physics loss. The residuals must equal 0
    rt = dNN2_dt2 + (mu/m)*dNN_dt + (k/m)*NN
    loss_phys = torch.mean(rt**2)

    # Initial conditions loss. x(0) = 1 and x'(0) = 0
    t0 = torch.tensor([0.0], requires_grad=True).double()
    NN_0 = model(t0)
    dNN_0 = torch.autograd.grad(NN_0, t0, create_graph=True)[0]
    loss_IC1 = torch.mean((NN_0 - 1)**2)
    loss_IC2 = torch.mean(dNN_0**2)

    # Data loss. The model must provide the same output as the closed form solution
    # Version 1
    # loss_data = []
    # for i in range(len(t_train)):
    #     data_loss_increment = torch.mean((model(t_train[i]) - y_train[i])**2)
    #     loss_data.append(data_loss_increment)
    # loss_data = torch.stack(loss_data).mean()
    # Version 2
    predictions = model(t_train)
    loss_fn = nn.MSELoss()
    
    assert predictions.shape == y_train.shape
    loss_data = loss_fn(predictions, y_train)
    
    lam_phys = 1e-4
    lam_ic = 1

    total_loss = loss_data + loss_phys * lam_phys + (loss_IC1 + loss_IC2) * lam_ic

    return total_loss


In [30]:
# print("Total loss shape: ", loss.shape)
# print("Total loss: ", loss.item())
# for i in range(10):
#     print(f'w({i}):', model[0].weight[i,0].item())
# Finite difference approximation

# w1_plus = model[0].weight.tolist()
# w1_minus = model[0].weight.tolist()
# b1 = model[0].bias.tolist()
# w2 = model[2].weight[0].tolist()
# b2 = model[2].bias.item()
e = 1e-5
fda_list_w1 = []
fda_list_w2 = []
l_plus_w1 = []
l_minus_w1 = []
l_plus_w2 = []
l_minus_w2 = []
loss_list_plus = []
loss_list_minus = []

for i in range(10):
    og_weight = model[0].weight[i,0].item()
    with torch.no_grad():
        model[0].weight[i, 0] = og_weight + e
    l_plus_w1.append(calc_loss().item())

    with torch.no_grad():
        model[0].weight[i, 0] = og_weight

    with torch.no_grad():
        model[0].weight[i, 0] = og_weight - e
    l_minus_w1.append(calc_loss().item())

    with torch.no_grad():
        model[0].weight[i, 0] = og_weight

    fda_list_w1.append((l_plus_w1[-1] - l_minus_w1[-1]) / (2 * e))

for i in range(10):
    og_weight = model[2].weight[0, i].item()
    with torch.no_grad():
        model[2].weight[0, i] = og_weight + e
    l_plus_w2.append(calc_loss().item())

    with torch.no_grad():
        model[2].weight[0, i] = og_weight

    with torch.no_grad():
        model[2].weight[0, i] = og_weight - e
    l_minus_w2.append(calc_loss().item())

    with torch.no_grad():
        model[2].weight[0, i] = og_weight

    fda_list_w2.append((l_plus_w2[-1] - l_minus_w2[-1]) / (2 * e))

print("W1 Plus losses:", l_plus_w1)
print("W1 Minus losses:", l_minus_w1)
print("W2 Plus losses:", l_plus_w2)
print("W2 Minus losses:", l_minus_w2)
print("W1 Numerical gradient: ", fda_list_w1)
print("W2 Numerical gradient: ", fda_list_w2)

W1 Plus losses: [4.676308603755611, 4.676300356770021, 4.676294964745519, 4.676298119328453, 4.676300681784793, 4.676301992146987, 4.676303426569265, 4.676301458301872, 4.676300800112701, 4.67630157223246]
W1 Minus losses: [4.676293469142493, 4.676301715870775, 4.676307108021712, 4.676303953352557, 4.676301390845971, 4.676300080500484, 4.67629864608273, 4.676300614339156, 4.676301272523586, 4.676300500397479]
W2 Plus losses: [4.6762214674784, 4.67622442287867, 4.676339796711774, 4.676286611556588, 4.6764045645455745, 4.676406826217896, 4.676224693746386, 4.67640743610759, 4.676242376846178, 4.676192828345772]
W2 Minus losses: [4.67638060629751, 4.676377650995466, 4.676262276445659, 4.676315462027551, 4.676197511729432, 4.676195249195745, 4.676377380002069, 4.676194640276006, 4.676359697212802, 4.6764092478783486]
W1 Numerical gradient:  [0.7567306559153052, -0.06795503768586286, -0.6071638096294407, -0.2917012051906198, -0.035453058888634814, 0.09558232516582164, 0.23902432677580518, 0

In [35]:
model.zero_grad()

total_loss = calc_loss()
total_loss.backward()

dW1 = model[0].weight.grad[:, 0].tolist()  # 10 first-layer gradients
dW2 = model[2].weight.grad[0, :].tolist()  # 10 second-layer gradients
print("dw1: ", dW1)
print("dw2: ", dW2)

dw1:  [0.7567306560059663, -0.06795503768767829, -0.6071638095835308, -0.29170120516640435, -0.03545305881757348, 0.095582325145213, 0.2390243267979792, 0.04219813578640437, -0.02362054431763276, 0.0535917490795042]
dw2:  [-7.9569409555152175, -7.661405839853138, 3.876013305748104, -1.44252354813245, 10.35264080714181, 10.578851107601555, -7.63431278417945, 10.63979157927649, -5.8660183312485685, -10.820976628811872]


In [42]:
dW1 = np.asarray(dW1)
dW2 = np.asarray(dW2)
fda_list_w1 = np.asarray(fda_list_w1)
fda_list_w2 = np.asarray(fda_list_w2)

absolute_error_w1 = np.abs(dW1 - fda_list_w1)
absolute_error_w2 = np.abs(dW2 - fda_list_w2)
fig = px.line(y = [absolute_error_w1, absolute_error_w2], title='Absolute gradient error')
fig.update_layout(xaxis_title='Weight index', yaxis_title='Absolute error', template='plotly_dark', 
                  width=800, height=500, showlegend=True)
fig.show()